# VibeShift Flow Matching Training Notebook

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from tqdm.notebook import tqdm

from training import Trainer, TrainingConfig
from dataloader import create_dataloader, GenreAwareLatentDataset



In [ ]:
SOURCE_DIR = "data/latent_data/latent_classical"
TARGET_DIR = "data/latent_data/latent_synth1"
BATCH_SIZE  = 64   # RTX 4090 24GB — 768/12/12 fits comfortably at bs=64
NUM_WORKERS = 4
MAX_SAMPLES = None
SHUFFLE     = True
DROP_LAST   = True
TRAIN_SPLIT = 0.8   # 80% train  → ~19,200 samples
VAL_SPLIT   = 0.1   # 10% val    →  ~2,400 samples
TEST_SPLIT  = 0.1   # 10% test   →  ~2,400 samples

# Import required utilities
from torch.utils.data import Subset, DataLoader
from dataloader import default_collate_with_dynamic_padding

# Load full dataset first
# genre_aware=True → uses GenreAwareLatentDataset so every sample carries
# genre_id=1 (target/punk) rather than silently defaulting to 0 (source/synth).
# Without this, all genre_ids are 0, genre-1 embedding is never trained,
# and inference with TARGET_GENRE_ID=1 produces garbage despite low training loss.
full_loader, full_dataset = create_dataloader(
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS,
    max_samples=MAX_SAMPLES,
    drop_last=DROP_LAST,
    pin_memory=True,
    persistent_workers=True,  # keeps workers alive between epochs — avoids Jupyter worker restarts
    genre_aware=True,         # all samples get genre_id=1 (target genre) by default
)

# Split into train / val / test  (80 / 10 / 10)
dataset_size = len(full_dataset)
train_size   = int(dataset_size * TRAIN_SPLIT)
val_size     = int(dataset_size * VAL_SPLIT)
test_size    = dataset_size - train_size - val_size   # absorb rounding remainder

print(f"Dataset: {dataset_size} samples")
print(f"  Train: {train_size} samples ({train_size/dataset_size:.0%})")
print(f"  Val:   {val_size}   samples ({val_size/dataset_size:.0%})")
print(f"  Test:  {test_size}  samples ({test_size/dataset_size:.0%})")

# Create index-based subsets
train_indices = list(range(train_size))
val_indices   = list(range(train_size, train_size + val_size))
test_indices  = list(range(train_size + val_size, dataset_size))

train_subset = Subset(full_dataset, train_indices)
val_subset   = Subset(full_dataset, val_indices)
test_subset  = Subset(full_dataset, test_indices)

# Build DataLoaders using the same collate function for dynamic padding
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS,
    drop_last=DROP_LAST,
    pin_memory=True,
    collate_fn=default_collate_with_dynamic_padding,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
    pin_memory=True,
    collate_fn=default_collate_with_dynamic_padding,
)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
    pin_memory=True,
    collate_fn=default_collate_with_dynamic_padding,
)

print(f"\nBatch size: {BATCH_SIZE}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val   batches per epoch: {len(val_loader)}")
print(f"Test  batches per epoch: {len(test_loader)}")

info = full_dataset.get_info()
print(f"Embedding dim: {info['embedding_dim']}")

# Validate data with sample batch
try:
    batch = next(iter(train_loader))
    print(f"\nBatch elements: {len(batch)}")

    x0_sample = batch[0]
    x1_sample = batch[1]
    genre_ids = None
    mask = None

    # Detect third element by dtype
    if len(batch) > 2:
        if batch[2].dtype in [torch.long, torch.int64]:
            genre_ids = batch[2]
            mask = batch[3] if len(batch) > 3 else None
        elif batch[2].dtype == torch.float32:
            mask = batch[2]

    print(f"[OK] Sample batch loaded")
    print(f"  x0 shape: {x0_sample.shape}, range: [{x0_sample.min():.4f}, {x0_sample.max():.4f}]")
    print(f"  x1 shape: {x1_sample.shape}, range: [{x1_sample.min():.4f}, {x1_sample.max():.4f}]")
    if genre_ids is not None:
        print(f"  genre_ids: {genre_ids.shape}, unique: {genre_ids.unique().tolist()}")
    if mask is not None:
        print(f"  mask shape: {mask.shape}, valid ratio: {mask.sum() / mask.numel():.2%}")

except Exception as e:

    print(f"[WARNING] Could not load sample batch: {e}")    traceback.print_exc()
    import traceback

In [ ]:
# ── Dataloader sanity check ───────────────────────────────────────────────────
_batch = next(iter(train_loader))

x0   = _batch[0]
x1   = _batch[1]

# Unpack genre_ids and mask depending on how many elements the batch has
genre_id = None
mask     = None
if len(_batch) > 2:
    if _batch[2].dtype in [torch.long, torch.int64]:
        genre_id = _batch[2]
        mask = _batch[3] if len(_batch) > 3 else None
    elif _batch[2].dtype == torch.float32:
        mask = _batch[2]

print(f"x0 shape  : {x0.shape}")
print(f"x1 shape  : {x1.shape}")
print(f"x0 vs x1 identical : {torch.allclose(x0, x1)}")
print(f"mean difference    : {(x1 - x0).abs().mean():.6f}")
print(f"x0 range  : {x0.min():.4f}  to  {x0.max():.4f}")
if genre_id is not None:
    print(f"genre_ids : {genre_id.shape}, unique={genre_id.unique().tolist()}")
else:
    print("genre_ids : not present in batch")
if mask is not None:
    print(f"mask active (mean) : {mask.mean():.3f}  ({mask.shape})")
else:
    print("mask      : not present in batch")


In [ ]:
import shutil

# ── Directory to save test-split files ───────────────────────────────────────
TEST_SAVE_DIR = "data/latent_data/test_split"   # ← change as needed

test_src_dir = Path(TEST_SAVE_DIR) / "source"
test_tgt_dir = Path(TEST_SAVE_DIR) / "target"
test_src_dir.mkdir(parents=True, exist_ok=True)
test_tgt_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving {len(test_indices)} test-split file pairs to: {TEST_SAVE_DIR}")

for idx in test_indices:
    src_file = full_dataset.source_files[idx]
    tgt_file = full_dataset.target_files[idx]
    shutil.copy2(src_file, test_src_dir / src_file.name)
    shutil.copy2(tgt_file, test_tgt_dir / tgt_file.name)

print(f"[OK] Test source files → {test_src_dir}  ({len(list(test_src_dir.glob('*.pt')))} files)")
print(f"[OK] Test target files → {test_tgt_dir}  ({len(list(test_tgt_dir.glob('*.pt')))} files)")


In [ ]:
try:
    from models.dit import DiT
    from models.flow import FlowMatching

   
    DIT_CONFIG = {
        "input_dim": 1024,
        "embed_dim": 512,
        "num_blocks": 8,
        "num_heads": 8,
        "num_genres": 2,
        "hidden_dim": 2048,  # explicit — do NOT let DiT fall back to dit.yaml (1024)
    }

    # Validate embedding dimension matches dataloader
    if info['embedding_dim'] != DIT_CONFIG['input_dim']:
        raise ValueError(f"Dimension mismatch! Dataset: {info['embedding_dim']}, Model: {DIT_CONFIG['input_dim']}")

    dit = DiT(**DIT_CONFIG)
    model = FlowMatching(dit)

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"[OK] Model loaded on {DEVICE}")
    print(f"  embed_dim={DIT_CONFIG['embed_dim']}, num_blocks={DIT_CONFIG['num_blocks']}, num_heads={DIT_CONFIG['num_heads']}")
    print(f"  Total parameters: {total_params:,}  (~{total_params/1e6:.0f} M)")
    print(f"  Genre embedding table: {DIT_CONFIG['num_genres'] + 1} slots (0=source, 1=target, 2=null/CFG)")
except ImportError as e:
    print(f"[WARNING] Could not import models: {e}")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = nn.Linear(512, 512).to(DEVICE)
except ValueError as e:
    print(f"[ERROR] {e}")
    raise


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPERIMENT_NAME = "vibeshift_flow_matching"

# ── Checkpoint directory (set to an absolute path on Vast.ai) ─────────────────
# Change this to wherever you want checkpoints saved, e.g.:
#   "/workspace/vibe/checkpoints"    ← inside the container
#   "/root/checkpoints"              ← persists across restarts if on root volume
CHECKPOINT_DIR = "/workspace/vibe/checkpoints"

# Ensure the directory exists before Trainer tries to write into it
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,          # PyTorch default is 1e-3 — 10× too high for this architecture
    weight_decay=1e-5,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=300,   # must match NUM_EPOCHS — LR decays to eta_min over full run
    eta_min=1e-6,
)

def create_loss_fn(model):
    """
    Create a loss function that wraps the model's loss computation.
    Detects capability once to avoid per-batch exception overhead.
    """
    has_compute_loss = hasattr(model, "compute_loss")

    def loss_fn(x0, x1, genre_ids, mask=None):
        if has_compute_loss:
            # FlowMatching path: supports mask argument
            return model.compute_loss(x0, x1, genre_ids, mask=mask)
        else:
            # Generic nn.Module path: no mask support
            return model(x0, x1, genre_ids)
    return loss_fn

loss_fn = create_loss_fn(model)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    name=EXPERIMENT_NAME,
    use_amp=True if DEVICE == "cuda" else False,
)

print(f"[OK] Trainer initialized")

print(f"Device: {DEVICE}")

print(f"AMP: {trainer.use_amp}")print(f"Checkpoint dir: {trainer.checkpoint_dir}")

In [ ]:
# ── Auto-resume from last epoch checkpoint ────────────────────────────────────
import re

ckpt_dir = Path(CHECKPOINT_DIR) / EXPERIMENT_NAME

def _parse_epoch(p: Path) -> int:
    """Return the epoch number embedded in checkpoint_epoch_N_epoch_N.pt, or -1."""
    m = re.search(r"checkpoint_epoch_(\d+)_epoch_\d+\.pt$", p.name)
    return int(m.group(1)) if m else -1

# Only consider periodic epoch checkpoints (ignore best.pt)
epoch_ckpts = [p for p in ckpt_dir.glob("checkpoint_epoch_*.pt") if _parse_epoch(p) >= 0]

if epoch_ckpts:
    # Pick the checkpoint with the highest epoch number
    latest_ckpt = max(epoch_ckpts, key=_parse_epoch)
    try:
        start_epoch = trainer.load_checkpoint(str(latest_ckpt), load_optimizer=True)
        print(f"[OK] Resumed from: {latest_ckpt.name}  (epoch {start_epoch})")
    except Exception as e:
        print(f"[WARNING] Failed to load '{latest_ckpt.name}': {e}")
        print("Starting fresh training instead.")
        start_epoch = 0
else:
    print(f"No epoch checkpoints found in {ckpt_dir} — starting fresh training.")
    start_epoch = 0


In [ ]:
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 30  # ~10% of total epochs — scales with longer run

remaining_epochs = NUM_EPOCHS - start_epoch
if remaining_epochs <= 0:
    print(f"[OK] Already completed {start_epoch}/{NUM_EPOCHS} epochs — nothing to train.")
else:
    print(f"Starting training: epochs {start_epoch + 1} → {NUM_EPOCHS}  ({remaining_epochs} remaining)")
    with tqdm(total=remaining_epochs, desc="Training Progress", unit="epoch") as pbar:
        results = trainer.train(
            train_dataloader=train_loader,
            num_epochs=remaining_epochs,
            loss_fn=loss_fn,
            val_dataloader=val_loader,
            scheduler=scheduler,
            gradient_clip=1.0,
            accumulation_steps=1,   # bs=64 directly on 4090 — no accumulation needed
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            log_interval=5,
            monitor_gradients=True,
            save_interval=10,
        )
        pbar.update(remaining_epochs)

    # Validate results dict
    assert isinstance(results, dict), "[ERROR] Training did not return results dict"
    assert 'epoch_losses' in results, "[ERROR] Missing epoch_losses in results"

    num_epochs_trained = len(results['epoch_losses'])

    print(f"\n[OK] Training completed!")
    print(f"  Epochs run this session: {num_epochs_trained}/{remaining_epochs}")
    print(f"  Total epochs done: {start_epoch + num_epochs_trained}/{NUM_EPOCHS}")
    print(f"  Train loss: {results['epoch_losses'][-1]:.6f}")
    if results.get('val_losses') and len(results['val_losses']) > 0:
        print(f"  Val loss: {results['val_losses'][-1]:.6f}")
    print(f"  Best loss: {results['best_loss']:.6f} (epoch {results['best_epoch']+1})")
    print(f"  Loss reduction: {results['epoch_losses'][0] - results['epoch_losses'][-1]:.6f}")
    print(f"  Gradient norms tracked: {len(results['gradient_norms'])}")
    print(f"  Checkpoints saved to: {trainer.checkpoint_dir}")


In [ ]:
# ── Test-set evaluation ──────────────────────────────────────────────────────
print("Evaluating on held-out test set...")
model.eval()
test_losses = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test batches"):
        x0 = batch[0].to(DEVICE)
        x1 = batch[1].to(DEVICE)
        genre_ids = None
        mask = None

        if len(batch) > 2:
            if batch[2].dtype in [torch.long, torch.int64]:
                genre_ids = batch[2].to(DEVICE)
                mask = batch[3].to(DEVICE) if len(batch) > 3 else None
            elif batch[2].dtype == torch.float32:
                mask = batch[2].to(DEVICE)

        loss = loss_fn(x0, x1, genre_ids, mask=mask)
        test_losses.append(loss.item())

avg_test_loss = sum(test_losses) / len(test_losses)
print(f"\n[OK] Test evaluation complete")
print(f"  Test loss (mean over {len(test_losses)} batches): {avg_test_loss:.6f}")
if results.get('val_losses') and results['val_losses']:
    print(f"  Val  loss (last epoch):  {results['val_losses'][-1]:.6f}")
print(f"  Train loss (last epoch): {results['epoch_losses'][-1]:.6f}")


In [ ]:
import matplotlib.pyplot as plt

if not isinstance(results, dict) or 'epoch_losses' not in results:
    print("[WARNING] Cannot visualize: results dict not available or incomplete")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(results['epoch_losses']) + 1)
    axes[0, 0].plot(epochs, results['epoch_losses'], linewidth=2)
    axes[0, 0].set_title('Training Loss per Epoch')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)
    
    if results.get('learning_rates') and len(results['learning_rates']) > 0:
        axes[0, 1].plot(epochs[:len(results['learning_rates'])], results['learning_rates'], linewidth=2, color='green')
        axes[0, 1].set_title('Learning Rate Schedule')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Learning Rate')
        axes[0, 1].grid(True, alpha=0.3)
    else:
        axes[0, 1].text(0.5, 0.5, 'No learning rate data', ha='center', va='center')
        axes[0, 1].set_title('Learning Rate Schedule')
        axes[0, 1].axis('off')
    
    if results.get('gradient_norms') and len(results['gradient_norms']) > 0:
        axes[1, 0].plot(results['gradient_norms'], linewidth=1, alpha=0.7, color='orange')
        axes[1, 0].set_title('Gradient Norms')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Norm')
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center')
        axes[1, 0].set_title('Gradient Norms')
        axes[1, 0].axis('off')
    
    loss_improvement = results['epoch_losses'][0] - results['epoch_losses'][-1]
    summary_text = f"Training Summary:\n\n"
    summary_text += f"Best Loss: {results['best_loss']:.6f}\n"
    summary_text += f"Best Epoch: {results['best_epoch']}\n"
    summary_text += f"Total Epochs: {len(results['epoch_losses'])}\n"
    summary_text += f"Final Loss: {results['epoch_losses'][-1]:.6f}\n"
    summary_text += f"Loss Improvement: {loss_improvement:.6f}"
    
    axes[1, 1].text(0.1, 0.9, summary_text,
                    transform=axes[1, 1].transAxes,
                    fontsize=11,
                    verticalalignment='top',
                    fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n[OK] Visualization complete")
    print(f"Best Loss: {results['best_loss']:.6f} (Epoch {results['best_epoch']})")
    print(f"Final Loss: {results['epoch_losses'][-1]:.6f}")


In [ ]:
# Save final model (best checkpoint)
best_checkpoints = sorted(trainer.checkpoint_dir.glob("best*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)

if best_checkpoints:
    best_checkpoint = best_checkpoints[0]
    
    # Load best weights into current model
    trainer.load_checkpoint(str(best_checkpoint), load_optimizer=False)
    
    final_path = trainer.checkpoint_dir / "final_model.pt"
    torch.save(model.state_dict(), final_path)
    print(f"[OK] Final (best) model saved to {final_path}")
    print(f"[OK] Based on checkpoint: {best_checkpoint.name}")
else:
    # Fallback: save current model if no best checkpoint found
    print(f"[WARNING] No best checkpoint found, saving current model weights")
    final_path = trainer.checkpoint_dir / "final_model.pt"
    torch.save(model.state_dict(), final_path)
    print(f"[OK] Model saved to {final_path}")

print(f"\n[OK] All checkpoints saved in: {trainer.checkpoint_dir}")
all_checkpoints = list(trainer.checkpoint_dir.glob("*.pt"))
print(f"Total checkpoint files: {len(all_checkpoints)}")

